# Modul 20: Fairer Modellvergleich und verantwortungsvolles Abschlussprojekt

    **Notebooktyp:** Übungs- und Bewertungsnotebook  
    **Vorlesungen dieses Moduls:** Fair vergleichen, Projekt umsetzen  
    **Erwarteter Schwierigkeitsgrad:** Fortgeschrittenes Integrations- und Entscheidungsprojekt  
    **Orientierungszeit:** etwa 180 bis 240 Minuten

    ## Überblick

    Sie planen und dokumentieren ein vollständiges Klassifikationsprojekt. Identische Splits, Baselines, Leistungs- und Ressourcenmetriken, reproduzierbare Artefakte, sichere Inferenz, Teilgruppenprüfungen, Verteilungsverschiebung, Fehleranalyse und eine Modellkarte werden zu einem nachvollziehbaren Abschlussworkflow verbunden.

    ## Verwendete Vorlesungsnotebooks

    Die Aufgaben wurden aus dem Inhalt beider Vorlesungen dieses Moduls abgeleitet:

    - `ML Für Anfänger - Record_Module_20A_20260723.ipynb`
- `ML Für Anfänger - Record_Module_20B_20260723.ipynb`

    ## Colab-Kompatibilität

    Dieses Notebook ist für die kostenlose Version von Google Colab ausgelegt. Die Daten sind eingebaut, synthetisch erzeugt oder öffentlich verfügbar. Modelle und Trainingsbudgets sind bewusst klein gehalten. Führen Sie die Zellen in der vorgegebenen Reihenfolge aus.

## Lernziele

    Nach der Bearbeitung sollen Sie:

    - Problem, Ziel, Datenherkunft, Spaltenbedeutung und mögliche Leakage vor der Modellierung dokumentieren.
- EDA, Split, Baseline und mehrere Modelle unter identischen Bedingungen verbinden.
- Leistung, Trainingszeit, Inferenzzeit, Komplexität und Artefaktgröße fair vergleichen.
- Modell, Vorverarbeitung, Metadaten und Referenztests reproduzierbar speichern und laden.
- Eingaben vor der Inferenz validieren und Fehler verständlich behandeln.
- Teilgruppenleistung und einfache Verteilungsverschiebungen untersuchen.
- Verbesserungen nur mit Trainings- und Validierungsdaten auswählen.
- Eine kompakte Modellkarte und einen technisch begründeten Projektbericht erstellen.

    ## Bewertete Fähigkeiten

    - Projektplanung, Datensteckbrief, EDA und Leakage-Prüfung
- faire Splits, Baselines, Modellvergleich und Ressourcenmessung
- Versionierung, Speichern, Laden und sichere Inferenz
- Teilgruppenmetriken, Verteilungsverschiebung und Modellkarte
- Fehleranalyse, Schwellenwertwahl und Abschlussbericht

## Arbeitsanweisungen

Bearbeiten Sie die Aufgaben in der angegebenen Reihenfolge. Schreiben Sie Ihren Code ausschließlich in die klar markierten Arbeitszellen. Ergänzen Sie nach jeder Aufgabe eine kurze fachliche Reflexion. Verwenden Sie das Testset nicht für Modellwahl oder Hyperparameterentscheidungen, sofern die Aufgabe dies nicht ausdrücklich als abschließenden Schritt verlangt.

- Führen Sie zuerst das gemeinsame Setup aus.
- Verändern Sie vorgegebene Splits und Seeds nur, wenn eine Aufgabe dies ausdrücklich erlaubt.
- Prüfen Sie Formen, Datentypen und Wertebereiche frühzeitig.
- Begründen Sie Modell-, Metrik- und Visualisierungsentscheidungen.
- Achten Sie auf Datenleckage und eine saubere Trennung von Training, Validierung und Test.

## Gemeinsames Setup

Führen Sie diese Zelle einmal aus, bevor Sie mit Aufgabe 1 beginnen.

In [ ]:
# Gemeinsames Setup für dieses Notebook
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import io
import json
import pickle
import platform
import time

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_columns", 50)
warnings.filterwarnings("ignore", category=FutureWarning)

cancer_20 = load_breast_cancer(as_frame=True)
project_data_20 = cancer_20.data.copy()

# Positive Klasse 1 bedeutet in diesem Projekt "maligne". Diese explizite
# Umcodierung macht Recall und Fehlertypen fachlich leichter lesbar.
project_data_20["malignant"] = (cancer_20.target == 0).astype(int)
project_data_20["record_id"] = np.arange(100000, 100000 + len(project_data_20))

# Eine synthetische Standortgruppe dient ausschließlich der technischen
# Teilgruppenanalyse. Sie ist kein geschütztes personenbezogenes Merkmal.
texture_median_20 = float(project_data_20["mean texture"].median())
project_data_20["acquisition_site"] = np.where(
    project_data_20["mean texture"] <= texture_median_20,
    "Site_A",
    "Site_B",
)

# Diese absichtlich ungeeignete Spalte simuliert eine Information, die
# erst nach der endgültigen Diagnose vorliegt und deshalb Leakage wäre.
project_data_20["diagnosis_after_review"] = project_data_20["malignant"]

feature_columns_20 = [str(name) for name in cancer_20.feature_names]
target_column_20 = "malignant"
group_column_20 = "acquisition_site"
leakage_columns_20 = ["diagnosis_after_review"]
identifier_columns_20 = ["record_id"]

print("Projektdataframe:", project_data_20.shape)
print("Positive Klasse 1 = maligne")

print("Setup abgeschlossen. Zufallsstartwert:", RANDOM_SEED)


## Aufgabe 1: Projekt planen, Daten beschreiben und Leakage erkennen

    Erstellen Sie den Daten- und Problemsteckbrief für das Abschlussprojekt.

1. Formulieren Sie Problem, Zielvariable, positive Klasse, Analyseeinheit und primäre Fehlerrisiken.
2. Erstellen Sie eine Tabelle mit Spaltenname, Datentyp, Rolle und kurzer Bedeutung für mindestens zehn Merkmale sowie Ziel, Gruppe, ID und die verdächtige Nachdiagnose-Spalte.
3. Markieren Sie Identifier und potenzielle Leakage-Spalten und begründen Sie deren Ausschluss aus den Modellmerkmalen.
4. Prüfen Sie Form, Duplikate, Fehlwerte, Klassenverteilung und Gruppenverteilung.
5. Visualisieren Sie zwei fachlich sinnvolle Merkmale nach Zielklasse und eine Korrelationsübersicht für eine kleine Merkmalsauswahl.
6. Formulieren Sie drei vorsichtige EDA-Befunde, ohne Kausalität zu behaupten.

> **Hinweis:** Fragen Sie für jede Spalte, ob ihr Wert zum realen Vorhersagezeitpunkt bereits verfügbar wäre.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 1

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Fragen Sie für jede Spalte, ob ihr Wert zum realen Vorhersagezeitpunkt bereits verfügbar wäre.

## Aufgabe 2: Baseline und Modelle unter identischen Bedingungen vergleichen

    Bauen Sie einen fairen Modellvergleich.

1. Trennen Sie zunächst 20 Prozent als Testset und danach 25 Prozent des verbleibenden Teils als Validierung. Verwenden Sie Stratifikation und feste Seeds.
2. Verwenden Sie ausschließlich `feature_columns_20` als Modellmerkmale.
3. Vergleichen Sie eine häufigste-Klasse-Baseline, logistische Regression mit Skalierung, einen Entscheidungsbaum und einen kleinen Random Forest.
4. Trainieren Sie alle Modelle auf demselben Training und bewerten Sie sie auf derselben Validierung.
5. Messen Sie Accuracy, Balanced Accuracy, Recall der malignen Klasse, ROC-AUC, Trainingszeit, Inferenzzeit pro Validierungsbatch und serialisierte Größe.
6. Wählen Sie das Modell anhand einer vorher festgelegten Regel: höchste Validierungs-Balanced-Accuracy, bei Gleichstand höherer maligner Recall, danach kleinere Artefaktgröße.
7. Bewerten Sie das ausgewählte Modell noch nicht auf dem Testset.

> **Hinweis:** Definieren Sie die Auswahlregel vor dem Blick auf das Testset.

In [ ]:
# Speichern Sie das ausgewählte Modell als selected_model_20 und den
# Namen als selected_model_name_20.

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 2

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Definieren Sie die Auswahlregel vor dem Blick auf das Testset.

## Aufgabe 3: Modellversion speichern, laden und Inferenz absichern

    Erstellen Sie ein reproduzierbares Modellartefakt für `selected_model_20`.

1. Speichern Sie Modell, erwartete Spaltenreihenfolge, Zielcodierung, Gruppenspalte, Seed, Schwellenwert, Paketversionen und Trainingszeitstempel in einem Dictionary.
2. Serialisieren Sie das Artefakt in `io.BytesIO` und laden Sie es neu.
3. Schreiben Sie eine Funktion `safe_predict_20`, die einen DataFrame erwartet und fehlende, zusätzliche oder falsch sortierte Spalten, Fehlwerte, unendliche Werte und leere Eingaben verständlich ablehnt.
4. Geben Sie Wahrscheinlichkeit und Klasse zurück.
5. Prüfen Sie zwölf Referenzzeilen vor und nach dem Laden auf identische Ergebnisse.
6. Demonstrieren Sie mindestens zwei kontrolliert abgefangene ungültige Eingaben.

> **Hinweis:** Validieren Sie das Schema vor jeder Modellmethode, nicht erst nach einem Fehler.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 3

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Validieren Sie das Schema vor jeder Modellmethode, nicht erst nach einem Fehler.

## Aufgabe 4: Teilgruppen, Verteilungsverschiebung und Modellkarte prüfen

    Bewerten Sie das ausgewählte Modell verantwortungsvoll, zunächst auf der Validierung.

1. Schreiben Sie eine Funktion, die pro `acquisition_site` Beispielzahl, positive Rate, Accuracy, Balanced Accuracy, malignen Recall, Präzision und Spezifität berechnet.
2. Wenden Sie sie auf die Validierungsdaten mit der Standardschwelle 0.5 an.
3. Berichten Sie die größte absolute Lücke zwischen den Gruppen für malignen Recall und Spezifität.
4. Simulieren Sie eine einfache Messverschiebung, indem Sie für `Site_B` in einer Kopie der Validierungsmerkmale drei ausgewählte Spalten um 1.50 Trainingsstandardabweichungen verringern.
5. Vergleichen Sie Gesamt- und Gruppenmetriken vor und nach der Verschiebung.
6. Erstellen Sie eine strukturierte Modellkarte mit Zweck, Daten, Metriken, Gruppenprüfung, Grenzen, Risiken und nicht vorgesehenen Anwendungen.

> **Hinweis:** Berechnen Sie Spezifität aus TN und FP und verwenden Sie für jede Gruppe dieselbe Schwelle.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 4

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Berechnen Sie Spezifität aus TN und FP und verwenden Sie für jede Gruppe dieselbe Schwelle.

## Aufgabe 5: Integration: Fehler analysieren, Schwelle verbessern und Projektbericht abschließen

    Schließen Sie das Projekt ab, ohne das Testset zur Optimierung zu verwenden.

1. Analysieren Sie falsch-negative Validierungsfälle des ausgewählten Modells und vergleichen Sie deren ausgewählte Merkmale mit korrekt erkannten malignen Fällen.
2. Untersuchen Sie Schwellenwerte von 0.10 bis 0.90 auf der Validierung.
3. Wählen Sie den höchsten Schwellenwert, der mindestens 95 Prozent malignen Recall auf der Validierung erreicht. Falls keiner dies schafft, wählen Sie den Schwellenwert mit dem höchsten Recall und danach höchster Balanced Accuracy.
4. Vergleichen Sie auf der Validierung Standardschwelle und neue Schwelle anhand von Recall, Spezifität, Präzision und Balanced Accuracy.
5. Aktualisieren Sie die Artefaktmetadaten mit der gewählten Schwelle.
6. Bewerten Sie **erst jetzt** das unveränderte Testset mit der finalen Schwelle und berichten Sie Gesamt- und Teilgruppenmetriken.
7. Erstellen Sie einen kompakten Projektbericht mit Problem, Daten, Methode, Auswahlregel, Baseline, finalen Ergebnissen, Ressourcen, Fehleranalyse, Grenzen und nächsten Schritten.

> **Hinweis:** Verwenden Sie das Testset erst, nachdem Modell und Schwelle vollständig festgelegt sind.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 5

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Verwenden Sie das Testset erst, nachdem Modell und Schwelle vollständig festgelegt sind.

## Abschluss und Selbstkontrolle

Prüfen Sie vor der Abgabe, ob alle Arbeitszellen ausgefüllt sind, das Notebook von oben nach unten ohne unerwartete Fehler läuft, alle Diagramme beschriftet sind und jede Reflexion Ihre Beobachtungen sowie mindestens eine mögliche Fehlerquelle enthält.

- Alle Aufgaben und Unterpunkte wurden bearbeitet.
- Verwendete Seeds und Datenpartitionen sind nachvollziehbar.
- Testdaten wurden nicht vorzeitig für Entscheidungen genutzt.
- Ergebnisse werden vorsichtig und fachlich begründet interpretiert.
- Es gibt keine hardcodierten lokalen Dateipfade oder privaten Zugangsdaten.